# Pick the best checkpoint and promote it for inference

Workflow for comparing the different trained runs (on the **server**), bringing the
winner down to your **local** machine, and promoting it into `checkpoints/pretrained/`
(the only checkpoint folder that is git-tracked).

**Where each step runs:**

| Step | Runs on | What it does |
|------|---------|--------------|
| 1 | server | Rank every run by best `val_loss` |
| 2 | server | Held-out eval of the top candidates (the real test) |
| 3 | local  | `scp` just the winner down |
| 4 | local  | Promote winner into `checkpoints/pretrained/`, then commit + push |
| 5 | local  | Run inference with the promoted weights |

Run all cells from the **project root** (`data_interpolation/`), not from `notebooks/`.

In [16]:
# Run once: move from notebooks/ up to the project root so relative paths work.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

cwd: /home/özkan/CAI-MedImg/data_interpolation


## Step 1 — (server) Rank all runs by validation loss

Scans every `checkpoints/*/history.json` and prints the best (lowest) `val_loss`
per run, sorted. Top line = best by training metric.

In [17]:
import json, glob, os

rows = []
for h in glob.glob("checkpoints/*/history.json"):
    run = os.path.basename(os.path.dirname(h))
    try:
        hist = json.load(open(h))
        vals = [e["val_loss"] for e in hist if e.get("val_loss") is not None]
        if vals:
            best = min(vals)
            last = hist[-1]
            rows.append((best, run, last.get("epoch"), last.get("train_loss")))
    except Exception as e:
        print(f"  ! skip {run}: {e}")

print(f"{'best_val_loss':>14}  {'run':<28} {'last_ep':>7} {'train_loss':>11}")
for best, run, ep, tr in sorted(rows):
    tr_s = tr if tr is None else f"{tr:.5f}"
    print(f"{best:>14.5f}  {run:<28} {ep!s:>7} {tr_s:>11}")

 best_val_loss  run                          last_ep  train_loss
       0.06894  pretrained                       100     0.06813
       0.06894  run_100ep                        100     0.06813
       0.06918  timing_2epoch_pretrained_v3        2     0.07074
       0.06919  bs16_timing                        2     0.07084


## Step 2 — (server) Confirm with held-out eval

`val_loss` can mislead if runs used different normalization or residual settings.
Validate the top 2–3 candidates on a real held-out BOLD file with `eval.py`.
Compare `mean_model_psnr` (higher = better) and `model_beats_naive_pct`.

Note: add `--residual` for any run trained with it — otherwise its scores are
meaningless. Check each run's README/config.

Edit `CANDIDATES` and `HELD_OUT_FILE` below, then run.

In [18]:
from pathlib import Path

files = sorted(Path("/srv/fMRI-data").rglob("*_bold.nii.gz"))
print(len(files))
print(files[:5])

BOLD_FILE = str(files[0])
print(BOLD_FILE)

28
[PosixPath('/srv/fMRI-data/sub-01_ses-03_task-HcpLanguage_dir-ap_bold.nii.gz'), PosixPath('/srv/fMRI-data/sub-01_ses-04_task-HcpSocial_dir-pa_bold.nii.gz'), PosixPath('/srv/fMRI-data/sub-02_ses-04_task-HcpEmotion_dir-pa_bold.nii.gz'), PosixPath('/srv/fMRI-data/sub-02_ses-05_task-HcpWm_dir-pa_bold.nii.gz'), PosixPath('/srv/fMRI-data/sub-04_ses-25_task-Stroop_dir-pa_bold.nii.gz')]
/srv/fMRI-data/sub-01_ses-03_task-HcpLanguage_dir-ap_bold.nii.gz


In [19]:
import sys
import subprocess
from pathlib import Path

CANDIDATES = ["run_100ep", "timing_2epoch_pretrained_v3"]

DATA_ROOT = Path("/srv/fMRI-data")
bold_files = sorted(DATA_ROOT.rglob("*_bold.nii.gz"))

print("found BOLD files:", len(bold_files))
print("first files:")
for f in bold_files[:5]:
    print(f)

HELD_OUT_FILE = str(bold_files[0])  # choose a real held-out file here
print("using held-out:", HELD_OUT_FILE)

RESIDUAL_RUNS = set()

for run in CANDIDATES:
    cmd = [
        sys.executable, "eval.py",
        "--weights", f"checkpoints/{run}/model_weights.pt",
        "--file", HELD_OUT_FILE,
        "--output-dir", f"results/eval_{run}",
        "--history", f"checkpoints/{run}/history.json",
    ]
    if run in RESIDUAL_RUNS:
        cmd.append("--residual")

    print(f"\n=== Evaluating: {run} ===")
    subprocess.run(cmd, check=True)

found BOLD files: 28
first files:
/srv/fMRI-data/sub-01_ses-03_task-HcpLanguage_dir-ap_bold.nii.gz
/srv/fMRI-data/sub-01_ses-04_task-HcpSocial_dir-pa_bold.nii.gz
/srv/fMRI-data/sub-02_ses-04_task-HcpEmotion_dir-pa_bold.nii.gz
/srv/fMRI-data/sub-02_ses-05_task-HcpWm_dir-pa_bold.nii.gz
/srv/fMRI-data/sub-04_ses-25_task-Stroop_dir-pa_bold.nii.gz
using held-out: /srv/fMRI-data/sub-01_ses-03_task-HcpLanguage_dir-ap_bold.nii.gz

=== Evaluating: run_100ep ===
=== Evaluation summary ===
  n_samples: 217
  file: /srv/fMRI-data/sub-01_ses-03_task-HcpLanguage_dir-ap_bold.nii.gz
  weights: checkpoints/run_100ep/model_weights.pt
  mean_model_l1: 0.0561900724429414
  mean_naive_l1: 0.05955833927964285
  mean_model_psnr: 27.028664700543448
  mean_naive_psnr: 26.57047834729161
  model_beats_naive_pct: 100.0
  loss_curve: results/eval_run_100ep/loss_curve.png

=== Evaluating: timing_2epoch_pretrained_v3 ===
=== Evaluation summary ===
  n_samples: 217
  file: /srv/fMRI-data/sub-01_ses-03_task-HcpLanguag

## Step 3 — (local) Bring just the winner down

Run this on your **local machine**. Edit the host and remote path. This copies only
the two files you need into `/tmp/` (we don't pollute the local `checkpoints/`).

## Step 4 — (local) Promote into `pretrained/` and commit

Overwrites the tracked `checkpoints/pretrained/` weights + history with the winner.
No `.gitignore` change needed — `pretrained/` is already whitelisted.

**Update `checkpoints/pretrained/README.md` by hand** after this if the winner's
architecture, normalization, residual flag, or training schedule differs from the
current one.

In [20]:
import shutil
from pathlib import Path

WINNER = "run_100ep"

src_dir = Path("checkpoints") / WINNER
dst_dir = Path("checkpoints/pretrained")
dst_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(src_dir / "model_weights.pt", dst_dir / "model_weights.pt")
shutil.copy(src_dir / "history.json", dst_dir / "history.json")

print(f"promoted {WINNER} into {dst_dir}")

promoted run_100ep into checkpoints/pretrained


## Step 5 — (local) Run inference with the promoted weights

Add `--residual` here **only** if the winner was trained with it.

In [21]:
from pathlib import Path

DATA_ROOT = Path("/srv/fMRI-data")
bold_files = sorted(DATA_ROOT.rglob("*_bold.nii.gz"))

print("found:", len(bold_files))
for f in bold_files[:5]:
    print(f)

INPUT_BOLD = str(bold_files[0])  # change this if you want another file
OUTPUT_FILE = "results/inference_2x.nii.gz"

print("input:", INPUT_BOLD)
print("output:", OUTPUT_FILE)

found: 28
/srv/fMRI-data/sub-01_ses-03_task-HcpLanguage_dir-ap_bold.nii.gz
/srv/fMRI-data/sub-01_ses-04_task-HcpSocial_dir-pa_bold.nii.gz
/srv/fMRI-data/sub-02_ses-04_task-HcpEmotion_dir-pa_bold.nii.gz
/srv/fMRI-data/sub-02_ses-05_task-HcpWm_dir-pa_bold.nii.gz
/srv/fMRI-data/sub-04_ses-25_task-Stroop_dir-pa_bold.nii.gz
input: /srv/fMRI-data/sub-01_ses-03_task-HcpLanguage_dir-ap_bold.nii.gz
output: results/inference_2x.nii.gz
